# 03 — Bronze: streets_chunk_1.csv via COPY INTO
First-time load of the 50% CSV chunk into a Bronze Delta table, with audit columns and table/column descriptions for discoverability.

## Step 1 — Widgets & Configuration

In [0]:
dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema")
dbutils.widgets.text("bronze_schema", "bronze", "3. Bronze Schema")
dbutils.widgets.text("chunks_volume", "chunks", "4. Chunks Volume")

CATALOG = dbutils.widgets.get("catalog_name")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
CHUNKS_VOL = dbutils.widgets.get("chunks_volume")

SOURCE_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{CHUNKS_VOL}/streets_chunk_1.csv"
TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.streets_csv_copyinto"

print(f"Source: {SOURCE_PATH}")
print(f"Target: {TABLE}")

## Step 0 — Pre-flight: confirm source chunk exists

In [0]:
try:
    dbutils.fs.ls(SOURCE_PATH)
except Exception:
    raise FileNotFoundError(
        f"{SOURCE_PATH} not found. Run 02_data_chunking first (Day 1)."
    )
print("Source chunk confirmed.")

## Step 1 — Create Bronze table (explicit schema, string columns + audit)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    noise       STRING COMMENT 'Cars on the street / street length — not a physical measure',
    pollution   STRING COMMENT 'Car pollution on the street / street length',
    date        STRING COMMENT 'Sample timestamp, ISO-8601 ms, one reading per street per 10s',
    light       STRING COMMENT 'Ambient light 0-100, depends on rain + hour of day',
    raining     STRING COMMENT 'Rain intensity 0-100%; observed range ~-1 to ~101 due to sensor noise — see assumptions doc',
    street_id   STRING COMMENT 'FK to streets_list.street_id (1-36)',
    load_dt     TIMESTAMP COMMENT 'Audit: load timestamp',
    source      STRING COMMENT 'Audit: source file name'
)
USING DELTA
COMMENT 'Bronze: first-time load of streets.csv chunk 1 (50%) via COPY INTO. Grain: (street_id, date).'
""")
print(f"Table ready: {TABLE}")

## Step 2 — COPY INTO (idempotent — already-loaded files are skipped)

In [0]:
copy_result = spark.sql(f"""
COPY INTO {TABLE}
FROM (
    SELECT
        noise, pollution, date, light, raining, street_id,
        current_timestamp() AS load_dt,
        _metadata.file_name AS source
    FROM '{SOURCE_PATH}'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'false', 'force' = 'false')
""")
display(copy_result)

## Step 3 — Sanity check (row count + audit columns populated)

In [0]:
row_count = spark.table(TABLE).count()
null_audit = spark.table(TABLE).filter("load_dt IS NULL OR source IS NULL").count()

print(f"Rows in {TABLE}: {row_count:,}")
print(f"Rows with missing audit columns: {null_audit}")

if row_count == 0:
    raise Exception("COPY INTO produced 0 rows — check source path/format.")
if null_audit > 0:
    raise Exception(f"{null_audit} rows missing load_dt/source — audit column logic broken.")

print("Sanity check PASSED.")